In [1]:
# show → code input visible by default
# hide-output → output hidden by default
# show hide-output → both (can combine on one line)

import pandas as pd
import numpy as np
#from google.cloud import bigquery
from common_lib.sql import BigQueryConnector
from common_lib.export import export_notebook_html
from common_lib.segmentation import (
    add_order_prefix,
    add_vlines_to_figure,
    add_segment_transition_columns,
    get_segment_movements_agg,
    plot_movement_charts,
)
import datetime as dt
import plotly.express as px


In [2]:
query_location = './sql/activity.sql'
parameters = {
    'start_date':'2026-01-01',
    'end_date': dt.datetime.now().strftime('%Y-%m-%d'),
    'exclude_networks':['']
}

bqc = BigQueryConnector()
cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

This query will process 8.51 GB when run.
Estimated query cost: $0.06


In [3]:
refresh_data = False

In [4]:
data = pd.DataFrame()

if refresh_data:
    data = bqc.get(query='./sql/activity.sql', is_path=True, query_parameters=parameters)
    data.to_pickle('./data/activity.pkl')
else:
    data = pd.read_pickle('./data/activity.pkl')


In [5]:

data

,user_id,dt,dt_week,dt_month,install_dt,install_dt_week,install_dt_month,days_since_install,loyalty_segment,payer_value_segment,payment_frequency_segment,payer_recency_segment,dsi_segment,daily_offers_avg_segment,daily_offers_count_segment,daily_offers_lastpurchase_segment,usd_net_iap_revenue,usd_net_ad_revenue
0,3916892338B266A9,2026-01-03,2025-12-28,2026-01-01,2026-01-01,2025-12-28,2026-01-01,2,0.0 (new install),0.0 (non-payer),0.0 (non-payer),0.0 (non-payer),D000-D006,0.00,0,none,NaN,NaN
1,16C50D1FE974F54D,2026-01-02,2025-12-28,2026-01-01,2025-10-03,2025-09-28,2025-10-01,91,1. 26-28 (dedicated),0.0 (non-payer),0.0 (non-payer),0.0 (non-payer),D091-D181,0.00,0,none,NaN,0.144432
2,3FB4C224D7237401,2026-01-02,2025-12-28,2026-01-01,2022-10-02,2022-10-02,2022-10-01,1188,2. 19-25 (frequent),0.0 (non-payer),0.0 (non-payer),0.0 (non-payer),D364+,0.00,0,none,NaN,0.037231
3,C97A8E18D65A01BF,2026-01-03,2025-12-28,2026-01-01,2024-09-17,2024-09-15,2024-09-01,473,1. 26-28 (dedicated),0.0 (non-payer),0.0 (non-payer),0.0 (non-payer),D364+,0.00,0,none,NaN,0.038291
4,DA43BED4930718F5,2026-01-02,2025-12-28,2026-01-01,2025-12-30,2025-12-28,2025-12-01,3,0.0 (new install),0.0 (non-payer),0.0 (non-payer),0.0 (non-payer),D000-D006,0.00,0,none,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20670865,FE5704C782706EFC,2026-08-16,2026-08-16,2026-08-01,2023-07-21,2023-07-16,2023-07-01,1122,1. 26-28 (dedicated),0.1 lapsed_payer,0.1 lapsed_payer,0.1 lapsed_payer,D364+,0.00,0,none,NaN,0.043745
20670866,E21CDC5B2BF07B1C,2026-08-16,2026-08-16,2026-08-01,2021-09-05,2021-09-05,2021-09-01,1806,1. 26-28 (dedicated),0.0 (non-payer),0.0 (non-payer),0.0 (non-payer),D364+,0.00,0,none,NaN,0.051479
20670867,F9DA913503853C5C,2026-08-16,2026-08-16,2026-08-01,2022-07-23,2022-07-17,2022-07-01,1485,3. 04-18 (moderate),0.0 (non-payer),0.0 (non-payer),0.0 (non-payer),D364+,0.00,0,none,NaN,0.016246
20670868,8FCC5636819AE1AF,2026-08-16,2026-08-16,2026-08-01,2023-06-29,2023-06-25,2023-06-01,1144,2. 19-25 (frequent),0.0 (non-payer),0.0 (non-payer),0.0 (non-payer),D364+,0.00,0,none,NaN,NaN


## Process data

In [6]:
data = add_order_prefix(data, ['daily_offers_avg_segment', 'daily_offers_count_segment', 'daily_offers_lastpurchase_segment'])
data[['daily_offers_avg_segment', 'daily_offers_count_segment', 'daily_offers_lastpurchase_segment']]

,daily_offers_avg_segment,daily_offers_count_segment,daily_offers_lastpurchase_segment
0,0_0.00,0_0,4_none
1,0_0.00,0_0,4_none
2,0_0.00,0_0,4_none
3,0_0.00,0_0,4_none
4,0_0.00,0_0,4_none
...,...,...,...
20670865,0_0.00,0_0,4_none
20670866,0_0.00,0_0,4_none
20670867,0_0.00,0_0,4_none
20670868,0_0.00,0_0,4_none


In [7]:
# Collapse "lapsed_payer" into the non-payer tier for all three payer segments — the SQL's own
# COALESCE default for a true never-paid user is '0.0 (non-payer)', and the dimension table's
# lapsed-payer label is '0.1 lapsed_payer' (identical string across all three columns), so a
# lapsed payer and a never-paid user end up on the same tier going forward. Flag columns keep
# the distinction available for anyone who wants it, done before the transition columns above
# so jump_size/duration treat lapsed-vs-non-payer as no shift, not a fake tier change.
LAPSED_LABEL = '0.1 lapsed_payer'
NON_PAYER_LABEL = '0.0 (non-payer)'

for col in ['payer_value_segment', 'payment_frequency_segment', 'payer_recency_segment']:
    data[f'{col}_is_lapsed'] = data[col] == LAPSED_LABEL
    data[col] = data[col].replace(LAPSED_LABEL, NON_PAYER_LABEL)




### Milestone dates

In [8]:
# Define vlines configuration
# Defines event timeline markers used for chart annotations
vlines_events = [
    # Timed Albums
    #{'x': pd.Timestamp('2025-11-04').timestamp() * 1000, 'annotation_text': 'JoesTastyTravels'},
    #{'x': pd.Timestamp('2025-12-16').timestamp() * 1000, 'annotation_text': 'WinterTales'},
    #{'x': pd.Timestamp('2026-01-26').timestamp() * 1000, 'annotation_text': 'AppletonLove'},
    #{'x': pd.Timestamp('2026-03-08').timestamp() * 1000, 'annotation_text': 'Diorama'},
    #{'x': pd.Timestamp('2026-04-18').timestamp() * 1000, 'annotation_text': 'ThroughTheAges'},
    #{'x': pd.Timestamp('2026-05-29').timestamp() * 1000, 'annotation_text': 'Appletonians'},
    #{'x': pd.Timestamp('2026-07-07').timestamp() * 1000, 'annotation_text': 'BitesizedAdventures'},
    #{'x': pd.Timestamp('2026-08-17').timestamp() * 1000, 'annotation_text': 'PlayTime'},
    #APS7
    {'x': pd.Timestamp('2026-06-15').timestamp() * 1000,'annotation_text': 'Step1Start'},
    #{'x': pd.Timestamp('2026-06-19').timestamp() * 1000,'annotation_text': 'AdsIssue_CA_AU'},
    {'x': pd.Timestamp('2026-06-29').timestamp() * 1000,'annotation_text': 'Step2Applied'},
    #{'x': pd.Timestamp('2026-07-08').timestamp() * 1000,'annotation_text': 'v0.79Rollout'},
    {'x': pd.Timestamp('2026-07-14').timestamp() * 1000,'annotation_text': 'BackToStep1'},
    #{'x': pd.Timestamp('2026-07-16').timestamp() * 1000,'annotation_text': 'FcapTo15_AllPlayers'},
    #{'x': pd.Timestamp('2026-08-04').timestamp() * 1000,'annotation_text': 'FcapTo30_AllPlayers'},
]

## Segment Transitions

For each user/day: what payment segment they were in the day before (previous *active*-day record — segment membership persists across inactive gaps, so a gap with no segment change is correctly not a shift), the jump size when they shift (rank difference over the segment's own tier order, positive = upgrade, negative = downgrade), and how long they'd been in the segment they just left (in both active days and calendar days), populated only on the row where the shift happens — 0 elsewhere, `NaN` only for each user's very first observed row (no prior state exists yet).

Starting with `payer_value_segment`; the function takes any segment column, so extending to `payment_frequency_segment`, `payer_recency_segment`, or `loyalty_segment` later is just another call.

(Logic lives in `add_segment_transition_columns` in `common_lib/segmentation.py`.)

## ARPDAU

In [9]:
data_arpdau_agg = data.groupby('dt').agg(
    usd_net_iap_revenue = ('usd_net_iap_revenue', 'sum'),
    usd_net_ad_revenue = ('usd_net_ad_revenue', 'sum'),
    user_id_count = ('user_id', 'nunique'),
).reset_index()

data_arpdau_agg['arpdau'] = (data_arpdau_agg['usd_net_iap_revenue'] + data_arpdau_agg['usd_net_ad_revenue']) / data_arpdau_agg['user_id_count'].replace(0, np.nan)
data_arpdau_agg

,dt,usd_net_iap_revenue,usd_net_ad_revenue,user_id_count,arpdau
0,2026-01-01,28755.059962,8882.558744,112469,0.334649
1,2026-01-02,44782.949123,9260.298230,116098,0.465497
2,2026-01-03,29171.675900,9704.887634,116528,0.333624
3,2026-01-04,24169.229485,10071.831181,118270,0.289516
4,2026-01-05,23864.662400,9528.951608,116350,0.287010
...,...,...,...,...,...
224,2026-08-13,20970.241279,5248.875815,63220,0.414728
225,2026-08-14,28461.692365,4946.680360,62606,0.533629
226,2026-08-15,17195.471099,4888.821777,61640,0.358279
227,2026-08-16,11814.066088,5012.197022,62348,0.269877


In [10]:
fig = px.line(data_arpdau_agg, 
              x='dt', 
              y=['arpdau'],
              #color='payer_value_segment',
              title='ARPDAU Over Time',
              width=1500,
              height=600,
              hover_data={'user_id_count': True, 'usd_net_iap_revenue': True, 'usd_net_ad_revenue': True},
              #barmode='group'
              )

fig = add_vlines_to_figure(fig, vlines_events)


fig.show()

## Movements

### `plot_movement_charts` reference

`show_counts` / `show_shares` (default on) draw the inflow (arrivals) dual-line charts. `show_outflow_counts` / `show_outflow_shares` (default **off** — turn on per call) draw the departures-side equivalents.

`metrics=[...]` accepts any of these column suffixes (each becomes its own one-line-per-tier chart):

- `balance_index`
- `net_share`
- `net_flow`
- `x_times_bigger`
- `ratio_upgrade_to_downgrade`
- `ratio_downgrade_to_upgrade`
- `log_ratio_u_to_d`
- `outflow_balance_index`
- `outflow_x_times_bigger`
- `share_ratio`
- `outflow_share_ratio`
- `outflow_net_share`
- `outflow_ratio_upgrade_to_downgrade`
- `outflow_ratio_downgrade_to_upgrade`
- `outflow_log_ratio_u_to_d`
- `jump_size_mean`, `shift_active_days_mean`, `shift_real_days_mean` (descriptive stats, not flow direction)

Not chartable as a line: `winner` / `outflow_winner` (text values, not numbers).

### Daily Offers

#### Average value

In [11]:
data_do_avg_seg = add_segment_transition_columns(data, 'daily_offers_avg_segment')

In [12]:
data_do_avg_movements_agg = get_segment_movements_agg('daily_offers_avg_segment', data_do_avg_seg)
data_do_avg_movements_agg

,dt,daily_offers_avg_segment,user_id_count,daily_offers_avg_segment_upgrades_count,daily_offers_avg_segment_downgrades_count,daily_offers_avg_segment_jump_size_mean,daily_offers_avg_segment_shift_active_days_mean,daily_offers_avg_segment_shift_real_days_mean,balance,ratio,...,daily_offers_avg_segment_upgrades_share,daily_offers_avg_segment_downgrades_share,daily_offers_avg_segment_net_share,daily_offers_avg_segment_balance_index,daily_offers_avg_segment_share_ratio,daily_offers_avg_segment_ratio_downgrade_to_upgrade,daily_offers_avg_segment_ratio_upgrade_to_downgrade,daily_offers_avg_segment_log_ratio_u_to_d,daily_offers_avg_segment_winner,daily_offers_avg_segment_x_times_bigger
0,2026-01-02,0_0.00,2,0,-2,-3.000000,1.000000,1.000000,-2,NaN,...,0.000000,0.005900,-0.005900,-1.000000,5.000000,5.000000,0.200000,-1.609438,Downgrades,5.000000
1,2026-01-02,1_0.01_1.99,73,73,0,1.000000,1.000000,1.000000,73,0.000000,...,0.215339,0.000000,0.215339,1.000000,0.006803,0.006803,147.000000,4.990433,Upgrades,147.000000
2,2026-01-02,2_2.00_3.99,104,104,0,1.980769,1.000000,1.000000,104,0.000000,...,0.306785,0.000000,0.306785,1.000000,0.004785,0.004785,209.000000,5.342334,Upgrades,209.000000
3,2026-01-02,3_4.00_7.99,69,69,0,2.942029,1.000000,1.000000,69,0.000000,...,0.203540,0.000000,0.203540,1.000000,0.007194,0.007194,139.000000,4.934474,Upgrades,139.000000
4,2026-01-02,4_8.00_14.99,62,61,-1,3.870968,1.000000,1.000000,60,-0.016393,...,0.179941,0.002950,0.176991,0.967742,0.024390,0.024390,41.000000,3.713572,Upgrades,41.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1588,2026-08-17,2_2.00_3.99,33,32,-1,1.484848,42.575758,55.727273,31,-0.031250,...,0.132231,0.004132,0.128099,0.939394,0.046154,0.046154,21.666667,3.075775,Upgrades,21.666667
1589,2026-08-17,3_4.00_7.99,43,41,-2,2.023256,39.790698,54.418605,39,-0.048780,...,0.169421,0.008264,0.161157,0.906977,0.060241,0.060241,16.600000,2.809403,Upgrades,16.600000
1590,2026-08-17,4_8.00_14.99,16,13,-3,1.937500,47.687500,56.812500,10,-0.230769,...,0.053719,0.012397,0.041322,0.625000,0.259259,0.259259,3.857143,1.349927,Upgrades,3.857143
1591,2026-08-17,5_15.00_24.99,10,4,-6,-0.200000,51.400000,54.200000,-2,-1.500000,...,0.016529,0.024793,-0.008264,-0.200000,1.444444,1.444444,0.692308,-0.367725,Downgrades,1.444444


In [13]:
plot_movement_charts(
    data_do_avg_movements_agg, 
    'daily_offers_avg_segment', 
    vlines_events=vlines_events, 
    arpdau_df=data_arpdau_agg, 
    overlay_arpdau=True, 
    title_prefix='Daily Offers Avg Segment', 
    show_counts=True, 
    show_shares=True, 
    metrics=['share_ratio', 'balance_index', 'x_times_bigger'],
    height=600,
    width=1200)

### All products

#### Payment value

In [14]:
# Start with payer_value_segment; call again with a different segment_col to extend
# (e.g. payment_frequency_segment, payer_recency_segment, loyalty_segment) — no other changes needed.
data_payment_value_seg = add_segment_transition_columns(data, 'payer_value_segment')

In [15]:
data_payment_value_movements_agg = get_segment_movements_agg('payer_value_segment', data_payment_value_seg)
data_payment_value_movements_agg

,dt,payer_value_segment,user_id_count,payer_value_segment_upgrades_count,payer_value_segment_downgrades_count,payer_value_segment_jump_size_mean,payer_value_segment_shift_active_days_mean,payer_value_segment_shift_real_days_mean,balance,ratio,...,payer_value_segment_upgrades_share,payer_value_segment_downgrades_share,payer_value_segment_net_share,payer_value_segment_balance_index,payer_value_segment_share_ratio,payer_value_segment_ratio_downgrade_to_upgrade,payer_value_segment_ratio_upgrade_to_downgrade,payer_value_segment_log_ratio_u_to_d,payer_value_segment_winner,payer_value_segment_x_times_bigger
0,2026-01-02,0.0 (non-payer),225,0,-225,-1.088889,1.000000,1.000000,-225,NaN,...,0.000000,0.094379,-0.094379,-1.000000,451.000000,451.000000,0.002217,-6.111467,Downgrades,451.000000
1,2026-01-02,1.$0.01-$9.99,588,320,-268,0.071429,1.000000,1.000000,52,-0.837500,...,0.134228,0.112416,0.021812,0.088435,0.837754,0.837754,1.193669,0.177031,Upgrades,1.193669
2,2026-01-02,2.$10.00-$19.99,593,320,-273,0.121417,1.000000,1.000000,47,-0.853125,...,0.134228,0.114513,0.019715,0.079258,0.853354,0.853354,1.171846,0.158581,Upgrades,1.171846
3,2026-01-02,3.$20.00-$39.99,452,336,-116,0.535398,1.000000,1.000000,220,-0.345238,...,0.140940,0.048658,0.092282,0.486726,0.346211,0.346211,2.888412,1.060707,Upgrades,2.888412
4,2026-01-02,4.$40.00-$79.99,291,215,-76,0.494845,1.000000,1.000000,139,-0.353488,...,0.090185,0.031879,0.058305,0.477663,0.354988,0.354988,2.816993,1.035670,Upgrades,2.816993
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1819,2026-08-17,3.$20.00-$39.99,158,86,-72,0.113924,9.367089,10.892405,14,-0.837209,...,0.086694,0.072581,0.014113,0.088608,0.838150,0.838150,1.193103,0.176558,Upgrades,1.193103
1820,2026-08-17,4.$40.00-$79.99,108,69,-39,0.314815,8.500000,9.361111,30,-0.565217,...,0.069556,0.039315,0.030242,0.277778,0.568345,0.568345,1.759494,0.565026,Upgrades,1.759494
1821,2026-08-17,5.$80.00-$139.99,66,32,-34,-0.015152,14.454545,15.393939,-2,-1.062500,...,0.032258,0.034274,-0.002016,-0.030303,1.061538,1.061538,0.942029,-0.059719,Downgrades,1.061538
1822,2026-08-17,6.$140.00-$349.99,37,29,-8,0.567568,9.891892,11.405405,21,-0.275862,...,0.029234,0.008065,0.021169,0.567568,0.288136,0.288136,3.470588,1.244324,Upgrades,3.470588


In [ ]:
plot_movement_charts(data_payment_value_movements_agg, 'payer_value_segment', vlines_events=vlines_events, arpdau_df=data_arpdau_agg)

#### Payment frequency

In [17]:
data_payment_frequency_seg = add_segment_transition_columns(data, 'payment_frequency_segment')

In [18]:
data_payment_frequency_movements_agg = get_segment_movements_agg('payment_frequency_segment', data_payment_frequency_seg)
data_payment_frequency_movements_agg

,dt,payment_frequency_segment,user_id_count,payment_frequency_segment_upgrades_count,payment_frequency_segment_downgrades_count,payment_frequency_segment_jump_size_mean,payment_frequency_segment_shift_active_days_mean,payment_frequency_segment_shift_real_days_mean,balance,ratio,...,payment_frequency_segment_upgrades_share,payment_frequency_segment_downgrades_share,payment_frequency_segment_net_share,payment_frequency_segment_balance_index,payment_frequency_segment_share_ratio,payment_frequency_segment_ratio_downgrade_to_upgrade,payment_frequency_segment_ratio_upgrade_to_downgrade,payment_frequency_segment_log_ratio_u_to_d,payment_frequency_segment_winner,payment_frequency_segment_x_times_bigger
0,2026-01-02,0.0 (non-payer),225,0,-225,-1.097778,1.000000,1.000000,-225,NaN,...,0.000000,0.103164,-0.103164,-1.000000,451.000000,451.000000,0.002217,-6.111467,Downgrades,451.000000
1,2026-01-02,1.00-01,667,311,-356,-0.067466,1.000000,1.000000,-45,-1.144695,...,0.142595,0.163228,-0.020633,-0.067466,1.144462,1.144462,0.873773,-0.134935,Downgrades,1.144462
2,2026-01-02,2.01-05,622,433,-189,0.459807,1.000000,1.000000,244,-0.436490,...,0.198533,0.086657,0.111875,0.392283,0.437140,0.437140,2.287599,0.827503,Upgrades,2.287599
3,2026-01-02,3.05-12,381,276,-105,0.461942,1.000000,1.000000,171,-0.380435,...,0.126547,0.048143,0.078404,0.448819,0.381555,0.381555,2.620853,0.963500,Upgrades,2.620853
4,2026-01-02,4.12-22,177,129,-48,0.457627,1.000000,1.000000,81,-0.372093,...,0.059147,0.022008,0.037139,0.457627,0.374517,0.374517,2.670103,0.982117,Upgrades,2.670103
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1591,2026-08-17,2.01-05,249,145,-104,0.216867,11.755020,15.413655,41,-0.717241,...,0.141051,0.101167,0.039883,0.164659,0.718213,0.718213,1.392344,0.330989,Upgrades,1.392344
1592,2026-08-17,3.05-12,172,103,-69,0.232558,11.930233,13.593023,34,-0.669903,...,0.100195,0.067121,0.033074,0.197674,0.671498,0.671498,1.489209,0.398245,Upgrades,1.489209
1593,2026-08-17,4.12-22,105,65,-40,0.238095,14.857143,15.304762,25,-0.615385,...,0.063230,0.038911,0.024319,0.238095,0.618321,0.618321,1.617284,0.480748,Upgrades,1.617284
1594,2026-08-17,5.22-45,38,31,-7,0.631579,11.578947,12.131579,24,-0.225806,...,0.030156,0.006809,0.023346,0.631579,0.238095,0.238095,4.200000,1.435085,Upgrades,4.200000


In [ ]:
plot_movement_charts(data_payment_frequency_movements_agg, 'payment_frequency_segment', vlines_events=vlines_events, arpdau_df=data_arpdau_agg)